# Joint simulation-calibrated inference

This tutorial combines the EACF statistic, spectral concentration, segment stability, and frequency-lag morphology in one held-out-calibrated decision. The small simulation count keeps the example quick; scientific runs should use substantially more realisations.

In [ ]:
import numpy as np

from urdr import (
    CoherentSignalConfig,
    SegmentSystematicConfig,
    SimulationConfig,
    calibrate_joint_detector,
    make_observing_window,
    simulate_time_series,
)

In [ ]:
window = make_observing_window(
    duration_days=0.8,
    cadence_seconds=120.0,
    gaps_days=((0.39, 0.41),),
)
simulation = SimulationConfig(
    white_noise_sigma=0.2,
    granulation_amplitude=0.1,
    numax_uhz=1000.0,
    delta_nu_uhz=100.0,
    envelope_width_uhz=400.0,
    oscillation_amplitude=1.8,
)
centres = np.linspace(700.0, 1300.0, 7)
delta_nu_grid = np.array([90.0, 100.0, 110.0])
segments = ((0.0, 0.4), (0.4, 0.8))

In [ ]:
detector = calibrate_joint_detector(
    window=window,
    simulation=simulation,
    centre_frequencies_uhz=centres,
    filter_width_uhz=500.0,
    delta_nu_grid_uhz=delta_nu_grid,
    segments_days=segments,
    coherent_contaminants={
        "single_line": CoherentSignalConfig(1000.0, 0.8),
        "harmonic_comb": CoherentSignalConfig(333.3, 0.8, harmonics=3),
    },
    segment_systematics={
        "variance_jump": [
            SegmentSystematicConfig(0.4, 0.8, amplitude_scale=4.0)
        ],
    },
    realizations=16,
    validation_fraction=0.25,
    target_false_positive_rate=0.25,
    max_lag_seconds=25_000.0,
    seed=42,
)
detector.validation

In [ ]:
target = simulate_time_series(
    window,
    simulation,
    np.random.default_rng(200),
    include_oscillations=True,
)
result = detector.detect(target)
{
    "detected": result.detected,
    "probability": result.detection_probability,
    "false_alarm_probability": result.false_alarm_probability,
    "false_alarm_interval": result.false_alarm_interval,
    "delta_nu_uhz": result.delta_nu_uhz,
    "flags": result.diagnostic_flags,
}

The diagnostic flags are explanatory rather than additional vetoes. Increase `realizations` for scientific use: a target false-positive rate of 1 per cent cannot be resolved reliably with only the 16 realisations used in this quick tutorial.